# Export: valence SD vs. vote SD per episode (for Excel)

One row per `Episode #`:
- `valence_sd`: **population** SD of comment valence (`manual_valence` if present, else `valence`)
- `vote_sd`: the `Standard Deviation` key from `python_source_map`

Writes `valence_vs_vote_sd.xlsx` (and a `.csv`). Plot `valence_sd` (X) against `vote_sd` (Y) in Excel.

In [ ]:
%pip install -q pymongo pandas numpy openpyxl

In [ ]:
import json
import numpy as np
import pandas as pd
from pymongo import MongoClient

EXCLUDE_NEUTRAL = True       # drop valence == 0 before taking the SD
EXCLUDE_FIRST_POST = True    # drop each episode's original post (earliest created_at)
MIN_COMMENTS = 2             # episodes with fewer non-neutral comments are left out (SD of 1 comment is trivially 0)

with open("config.json") as f:
    config = json.load(f)
db = MongoClient(config["mongo_uri"])["consensus"]

# votes
sm = pd.DataFrame(list(db["python_source_map"].find({}, {"Episode #": 1, "Standard Deviation": 1, "Topic / Proposal Title": 1, "Choice": 1})))
sm = sm.drop_duplicates("Episode #").rename(columns={"Standard Deviation": "vote_sd", "Topic / Proposal Title": "title", "Choice": "choice"})
sm["vote_sd"] = pd.to_numeric(sm["vote_sd"], errors="coerce")
for col in ("title", "choice"):          # tolerate a missing field
    if col not in sm:
        sm[col] = np.nan

# comments
cm = pd.DataFrame(list(db["python_source_map_comments"].find({}, {"Episode #": 1, "created_at": 1, "valence": 1, "manual_valence": 1})))
for c in ("valence", "manual_valence"):
    cm[c] = pd.to_numeric(cm[c], errors="coerce") if c in cm else np.nan
cm["v"] = cm["manual_valence"].where(cm["manual_valence"].notna(), cm["valence"])     # manual wins (a manual 0 counts)

first = cm[cm["created_at"].notna()].sort_values("created_at").groupby("Episode #").head(1).index
use = cm[cm["v"].notna()]
if EXCLUDE_FIRST_POST:
    use = use[~use.index.isin(first)]
if EXCLUDE_NEUTRAL:
    use = use[use["v"] != 0]

per_ep = use.groupby("Episode #")["v"].agg(valence_sd=lambda s: s.std(ddof=0), n_comments="count").reset_index()
per_ep = per_ep[per_ep["n_comments"] >= MIN_COMMENTS]

out = per_ep.merge(sm[["Episode #", "title", "choice", "vote_sd"]], on="Episode #", how="inner").dropna(subset=["vote_sd"])
out = out[["Episode #", "title", "choice", "valence_sd", "vote_sd", "n_comments"]].sort_values("Episode #")
print(f"{len(out)} episodes exported")
out.head(10)

In [ ]:
out.to_excel("valence_vs_vote_sd.xlsx", index=False)
out.to_csv("valence_vs_vote_sd.csv", index=False)
print("Saved valence_vs_vote_sd.xlsx and valence_vs_vote_sd.csv in the notebook's folder")